# 분자 특징 생성 & 분석 — 모델링 전에 특징을 만들고 이해하기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fourmodern/2025_aidrugdiscovery/blob/main/20260825/notebooks/01_feature_engineering_analysis.ipynb)

**AI 신약개발 실습 · 시리즈 1편(모델링 이전 단계)**

모델을 학습하기 **전에**, 분자를 컴퓨터가 다룰 수 있는 **숫자 특징(feature)** 으로 바꾸고, 그 특징들이 서로/타깃과 어떻게 관련되는지 **탐색적으로 이해(EDA)** 하고, 쓸모없는 특징을 **걸러내는(filtering)** 과정을 다룹니다. 좋은 특징이 없으면 어떤 모델도 잘 배우지 못합니다.

이 노트북에서 하는 일:

1. **특징 생성(Feature Generation)**
   - **RDKit 분자기술자**: MW·logP·TPSA·HBD·HBA·RotB·방향족고리·고리·FracCSP3·중원자·이종원자·MolMR 등 해석 가능한 물성/위상 값
   - **Morgan/ECFP 지문**: 반경 2·2048비트 원형 부분구조 지문(ECFP4 상당)
2. **특징 분석(EDA)** — 여기가 이 노트북의 핵심
   - 기술자 **분포**(히스토그램 그리드), 기술자–**타깃(logS)** 관계, 기술자 간 **상관 히트맵**(공선성)
   - **화학공간** PCA/UMAP 2D 투영(표준화 후, logS 색상), **ECFP 비트 통계**(on-bit 빈도·희소성)
   - (선택) **Lipinski** 물성 요약
3. **특징 필터링(Feature Filtering)**
   - **저분산 필터**(`VarianceThreshold`) + **공선성 필터**(|r|>0.9 제거) → ECFP 차원 2048 → N 축소
   - 필터는 **train에서만 학습**(데이터 누수 방지)

> ⚠️ **무-날조**: ESOL은 실제 측정 데이터이며, 모든 통계·차원 수·그림은 이 노트북이 **실제로 계산**한 값입니다. 여기서 `logS`는 특징을 이해하기 위한 **분석용 색상/상관 대상일 뿐, 모델 학습은 하지 않습니다**(모델링은 다음 02 노트북).
>
> 🖥️ **실행 환경**: CPU만으로 충분합니다(딥러닝 없음). ESOL은 약 1,100개 분자로 수십 초 내 전 과정이 끝납니다.

## 0. 설치 & 환경 설정

In [ ]:
!pip install -q rdkit scikit-learn
# UMAP은 선택 사항(없어도 PCA로 대체). 설치 실패해도 진행되도록 조용히 시도.
!pip install -q umap-learn > /dev/null 2>&1 || true

# 한글 폰트(그래프 라벨 깨짐 방지)
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1 || true

import matplotlib as mpl, matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import seaborn as sns

_kf = [f for f in fm.findSystemFonts() if "Nanum" in f]
for _f in _kf:
    fm.fontManager.addfont(_f)

# seaborn 테마를 먼저 적용한 뒤, 한글 폰트 설정을 '복원'(테마가 폰트를 덮어쓰므로 순서 중요)
sns.set_theme(style="whitegrid", context="notebook")
if _kf:
    mpl.rcParams["font.family"] = "NanumGothic"     # sns.set_theme 이후 복원
mpl.rcParams["axes.unicode_minus"] = False          # 마이너스 기호 깨짐 방지
mpl.rcParams["figure.dpi"] = 120                    # 선명한 그래프(dpi>=120)

# colorblind-safe 팔레트(Okabe-Ito)
CB = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#56B4E9", "#CC79A7", "#F0E442", "#999999"]
sns.set_palette(CB)

import platform
print("python", platform.python_version(), "| 한글폰트:", "OK" if _kf else "기본(미검출)")


## 1. 데이터 — ESOL 수용해도

Delaney(2004)의 **측정 수용해도**(logS, log mol/L)와 각 분자의 SMILES입니다.
RDKit로 파싱되는 **유효 분자만** 사용합니다. `logS`는 이 노트북에서 **특징을 이해하기 위한 분석 대상**으로만 쓰이며, 모델을 학습하지는 않습니다.

In [ ]:
import pandas as pd, numpy as np
from rdkit import Chem
from rdkit import RDLogger; RDLogger.DisableLog("rdApp.*")

URL = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/delaney-processed.csv"
df = pd.read_csv(URL)
df = df.rename(columns={"measured log solubility in mols per litre": "logS"})[["smiles", "logS"]]

df["mol"] = df["smiles"].apply(Chem.MolFromSmiles)
n_raw = len(df)
df = df[df["mol"].notnull()].reset_index(drop=True)     # RDKit 유효 분자만
y = df["logS"].values

print(f"원본 행: {n_raw} | RDKit 유효 분자: {len(df)} (무효 {n_raw - len(df)}개 제거)")
print("logS 범위: %.2f ~ %.2f (평균 %.2f)" % (y.min(), y.max(), y.mean()))
df[["smiles", "logS"]].head()


## 2. 특징 생성 ① — RDKit 분자기술자

물성·위상 기반의 **해석 가능한** 소수 차원 특징을 계산합니다. 각 값은 분자 구조에서 직접 유도되는 물리화학적 의미를 가집니다(Todeschini & Consonni, 분자기술자 핸드북).

| 기술자 | 의미 |
|---|---|
| MW | 분자량 |
| logP | 소수성(Crippen) |
| TPSA | 위상 극성 표면적 |
| HBD / HBA | 수소결합 주개/받개 수 |
| RotB | 회전 가능 결합 수(유연성) |
| AromRings / Rings | 방향족 고리 / 전체 고리 수 |
| FracCSP3 | sp³ 탄소 비율(입체 포화도) |
| HeavyAtoms / Heteroatoms | 중원자 / 이종(비탄소·비수소)원자 수 |
| MolMR | 몰 굴절률(Crippen, 분극성/부피) |

In [ ]:
from rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors

def descriptors(m):
    return [
        Descriptors.MolWt(m), Crippen.MolLogP(m), rdMolDescriptors.CalcTPSA(m),
        Lipinski.NumHDonors(m), Lipinski.NumHAcceptors(m),
        rdMolDescriptors.CalcNumRotatableBonds(m), rdMolDescriptors.CalcNumAromaticRings(m),
        rdMolDescriptors.CalcNumRings(m), Descriptors.FractionCSP3(m),
        m.GetNumHeavyAtoms(), rdMolDescriptors.CalcNumHeteroatoms(m), Crippen.MolMR(m),
    ]
DESC_NAMES = ["MW", "logP", "TPSA", "HBD", "HBA", "RotB", "AromRings",
              "Rings", "FracCSP3", "HeavyAtoms", "Heteroatoms", "MolMR"]

X_desc = np.array([descriptors(m) for m in df["mol"]], dtype=float)
desc_df = pd.DataFrame(X_desc, columns=DESC_NAMES)
desc_df["logS"] = y
print("기술자 행렬:", X_desc.shape, f"({len(DESC_NAMES)}개 기술자)")
desc_df.describe().round(2)


## 3. 특징 생성 ② — Morgan/ECFP 지문 (radius=2, 2048비트)

각 원자 주변의 원형 부분구조를 해싱해 **2048비트 이진 벡터**로 만듭니다(반경 2 = ECFP4 상당; Rogers & Hahn 2010). 고차원이지만 대부분의 비트가 **거의 항상 0**입니다 → §5·§6에서 통계로 확인하고 필터링합니다.

In [ ]:
from rdkit.Chem import rdFingerprintGenerator

mfp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)   # ECFP4 상당
X_fp = np.array([list(mfp.GetFingerprint(m)) for m in df["mol"]], dtype=np.int8)

print("ECFP 행렬:", X_fp.shape)
print("전체 비트 중 1(on)의 평균 비율: %.3f%% → 매우 희소(sparse)" % (100 * X_fp.mean()))
print("분자당 켜진 비트 수: 평균 %.1f개 (2048 중)" % X_fp.sum(axis=1).mean())


## 4. 특징 분석(EDA) ① — 기술자 분포 & 타깃(logS) 관계

- **분포 히스토그램 그리드**: 각 기술자의 값 범위·치우침(skew)·이상치를 한눈에 봅니다. 왜곡이 큰 특징은 이후 스케일링/변환의 후보가 됩니다.
- **기술자–logS 상관 막대**: 각 기술자가 용해도와 얼마나(어느 방향으로) 관련되는지. logP·MW가 음의 상관(소수성·큰 분자일수록 덜 녹음)을 보이는지 등 화학 직관과 맞는지 확인합니다.

In [ ]:
# (1) 기술자 분포 히스토그램 그리드
ncol = 4
nrow = int(np.ceil(len(DESC_NAMES) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3 * nrow))
axes = axes.ravel()
for i, name in enumerate(DESC_NAMES):
    axes[i].hist(desc_df[name], bins=30, color=CB[i % len(CB)], edgecolor="white", linewidth=0.4)
    axes[i].set_title(name, fontsize=11)
    axes[i].set_xlabel("값"); axes[i].set_ylabel("분자 수")
for j in range(len(DESC_NAMES), len(axes)):
    axes[j].axis("off")
fig.suptitle("기술자 분포 (ESOL 실데이터)", fontsize=14, y=1.005)
plt.tight_layout(); plt.show()


In [ ]:
# (2) 기술자 vs logS 상관(피어슨) 막대 — 방향/세기
corr_y = desc_df[DESC_NAMES].corrwith(desc_df["logS"]).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
bar_colors = ["#D55E00" if v < 0 else "#0072B2" for v in corr_y.values]  # 음=주황, 양=파랑
ax.barh(corr_y.index, corr_y.values, color=bar_colors, edgecolor="black", linewidth=0.4)
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("logS와의 피어슨 상관계수 r")
ax.set_title("기술자–타깃(logS) 상관\n(음: 소수성/크기 ↑ → 덜 녹음, 양: 극성 ↑ → 잘 녹음)")
for yi, v in enumerate(corr_y.values):
    ax.text(v + (0.01 if v >= 0 else -0.01), yi, f"{v:.2f}",
            va="center", ha="left" if v >= 0 else "right", fontsize=9)
plt.tight_layout(); plt.show()

print("logS와 상관이 가장 강한 기술자 Top 3 (절댓값):")
print(corr_y.abs().sort_values(ascending=False).head(3).round(3).to_string())


## 5. 특징 분석(EDA) ② — 상관 히트맵(공선성) & 화학공간(PCA/UMAP)

- **상관 히트맵**: 기술자들끼리 강하게 겹치는(공선성) 부분을 봅니다. 예를 들어 MW·HeavyAtoms·MolMR은 서로 강한 양의 상관을 가질 수 있습니다 → 필터링/차원축소의 근거.
- **화학공간 투영**: 기술자를 **표준화**한 뒤 PCA(및 가능하면 UMAP)로 **2D**에 투영하고 **logS로 색칠**합니다. 색이 공간에서 매끄럽게 변하면, 이 기술자 표현이 이미 용해도 관련 구조를 담고 있다는 신호입니다.

In [ ]:
# (3) 기술자 상관 히트맵
corr_mat = desc_df[DESC_NAMES].corr()
fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(corr_mat, annot=True, fmt=".2f", cmap="vlag", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={"label": "피어슨 상관계수 r"}, annot_kws={"size": 8}, ax=ax)
ax.set_title("기술자 간 상관 히트맵 — 공선성 확인")
plt.tight_layout(); plt.show()

# 강한 공선성 쌍(|r|>0.9) 보고
import itertools
pairs = [(a, b, corr_mat.loc[a, b]) for a, b in itertools.combinations(DESC_NAMES, 2)
         if abs(corr_mat.loc[a, b]) > 0.9]
if pairs:
    print("강한 공선성(|r|>0.9) 기술자 쌍:")
    for a, b, r in sorted(pairs, key=lambda t: -abs(t[2])):
        print(f"  {a} ~ {b}: r={r:.3f}")
else:
    print("기술자 사이에 |r|>0.9 쌍은 없습니다.")


In [ ]:
# (4) 화학공간 2D 투영 — 표준화 후 PCA (+ 가능하면 UMAP), 색=logS
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

Xd_std = StandardScaler().fit_transform(X_desc)          # 표준화(기술자 스케일 상이)

pca = PCA(n_components=2, random_state=42)
emb_pca = pca.fit_transform(Xd_std)
evr = pca.explained_variance_ratio_
proj = [("PCA", emb_pca, f"PC1 ({evr[0]*100:.0f}%)", f"PC2 ({evr[1]*100:.0f}%)")]

# UMAP은 있으면 사용(비선형 구조), 없으면 PCA만
try:
    from umap import UMAP
    emb_umap = UMAP(n_components=2, random_state=42).fit_transform(Xd_std)
    proj.append(("UMAP", emb_umap, "UMAP-1", "UMAP-2"))
except Exception as e:
    print("UMAP 미설치/미사용 → PCA만 표시 (%s)" % type(e).__name__)

fig, axes = plt.subplots(1, len(proj), figsize=(6.4 * len(proj), 5.2), squeeze=False)
for ax, (title, emb, xl, yl) in zip(axes[0], proj):
    sc = ax.scatter(emb[:, 0], emb[:, 1], c=y, cmap="viridis", s=16, alpha=0.75,
                    edgecolor="k", linewidth=0.2)
    ax.set_title(f"화학공간 {title} (기술자 표준화, 색=logS)")
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    plt.colorbar(sc, ax=ax, fraction=0.046, label="logS")
plt.tight_layout(); plt.show()


## 6. 특징 분석(EDA) ③ — ECFP 비트 통계 (희소성)

2048비트 각각이 **몇 %의 분자에서 켜지는지**(on-bit 빈도)를 봅니다. 대부분의 비트는 거의 항상 0이라 정보가 없고(상수에 가까움), 일부만 자주 등장합니다. 이 **희소성**이 §7 필터링의 직접적 근거입니다.

In [ ]:
# 비트별 on 빈도(= 각 비트가 1인 분자 비율)
bit_freq = X_fp.mean(axis=0)          # 길이 2048
n_dead = int((bit_freq == 0).sum())   # 한 번도 켜지지 않은 비트
n_rare = int((bit_freq < 0.01).sum()) # 1% 미만 분자에서만 켜지는 비트

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
# (좌) on-bit 빈도 히스토그램(로그 y) — 대부분 0 근처
axes[0].hist(bit_freq, bins=60, color="#E69F00", edgecolor="white", linewidth=0.4)
axes[0].set_yscale("log")
axes[0].set_xlabel("비트 on 빈도 (해당 비트=1인 분자 비율)")
axes[0].set_ylabel("비트 수 (log)")
axes[0].set_title("ECFP on-bit 빈도 분포 — 대부분 0 근처(희소)")

# (우) 상위 빈출 비트 30개 막대
top = np.argsort(bit_freq)[::-1][:30]
axes[1].bar(range(30), bit_freq[top] * 100, color="#009E73", edgecolor="black", linewidth=0.3)
axes[1].set_xlabel("빈출 상위 비트 순위(1~30)")
axes[1].set_ylabel("on 빈도 (%)")
axes[1].set_title("가장 자주 켜지는 ECFP 비트 Top 30")
plt.tight_layout(); plt.show()

print(f"한 번도 켜지지 않은 비트: {n_dead} / 2048 ({100*n_dead/2048:.1f}%)")
print(f"1% 미만 분자에서만 켜지는 희소 비트: {n_rare} / 2048 ({100*n_rare/2048:.1f}%)")
print("→ 대부분의 비트는 거의 상수(0). 저분산 필터로 걸러낼 대상입니다.")


## 6-1. (선택) Lipinski 물성 요약

경구 약물유사성의 관례적 기준인 **Lipinski's Rule of Five**(MW≤500, logP≤5, HBD≤5, HBA≤10)를 얼마나 위반하는지 요약합니다. 특징(물성) 관점에서 데이터셋 분자들의 '약물스러움' 분포를 파악하는 용도입니다.

In [ ]:
ro5 = pd.DataFrame({
    "MW>500":  desc_df["MW"]  > 500,
    "logP>5":  desc_df["logP"] > 5,
    "HBD>5":   desc_df["HBD"] > 5,
    "HBA>10":  desc_df["HBA"] > 10,
})
violations = ro5.sum(axis=1)
print("Lipinski Rule of Five — 위반 개수 분포:")
print(violations.value_counts().sort_index().rename("분자 수").to_string())
print(f"\n위반 0개(모두 만족) 분자: {int((violations == 0).sum())} / {len(df)} "
      f"({100*(violations == 0).mean():.1f}%)")

fig, ax = plt.subplots(figsize=(6.5, 4))
vc = violations.value_counts().sort_index()
ax.bar(vc.index.astype(str), vc.values, color="#0072B2", edgecolor="black", linewidth=0.4)
ax.set_xlabel("Ro5 위반 규칙 수"); ax.set_ylabel("분자 수")
ax.set_title("Lipinski Rule of Five 위반 분포")
for xi, v in zip(vc.index.astype(str), vc.values):
    ax.text(xi, v + 3, str(v), ha="center", fontsize=9)
plt.tight_layout(); plt.show()


## 7. 특징 필터링 — ECFP 지문 다이어트 (train에서만 학습)

§6에서 확인한 희소성/중복을 실제로 제거합니다. **핵심 원칙: 필터는 train에서만 학습**해 test에 적용합니다(테스트 정보가 특징 선택에 새어들면 성능이 부풀려집니다 = 데이터 누수). 여기서는 모델을 학습하지 않지만, 특징 선택 자체도 train 기준으로 하는 습관을 그대로 보여 줍니다.

1. **저분산 필터** `VarianceThreshold(threshold=0.01)` — 이진 비트의 분산은 `p(1-p)`. 임계 0.01은 대략 `p<1%` 또는 `p>99%`인 **거의 상수 비트**를 제거합니다.
2. **공선성 필터** — 남은 비트들의 상관행렬 **상삼각**에서 `|r|>0.9`인 쌍은 정보가 중복되므로 한쪽 열을 드롭합니다.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold

# train/test 분할 — 필터 '학습'은 train에서만(누수 방지). (모델 학습은 없음)
idx = np.arange(len(df))
idx_tr, idx_te = train_test_split(idx, test_size=0.2, random_state=42)
Xtr_fp, Xte_fp = X_fp[idx_tr].astype(float), X_fp[idx_te].astype(float)
print("train:", len(idx_tr), "| test:", len(idx_te))

# (1) 저분산 필터 — train에서만 fit
vt = VarianceThreshold(threshold=0.01).fit(Xtr_fp)
Xtr_v, Xte_v = vt.transform(Xtr_fp), vt.transform(Xte_fp)
d_raw, d_var = X_fp.shape[1], Xtr_v.shape[1]
print(f"[저분산] {d_raw} → {d_var} 비트 ({d_raw - d_var}개 제거, 거의 상수 비트)")

# (2) 공선성 필터 — train 상관행렬 상삼각 |r|>0.9 열 드롭
corr = np.abs(np.corrcoef(Xtr_v, rowvar=False))
corr = np.nan_to_num(corr)                # 혹시 남은 상수열 방어
upper = np.triu(corr, k=1)                # 상삼각(대각 제외)
drop_mask = (upper > 0.9).any(axis=0)     # 앞선 어떤 열과 |r|>0.9면 드롭
keep = ~drop_mask
Xtr_filt, Xte_filt = Xtr_v[:, keep], Xte_v[:, keep]
d_filt = Xtr_filt.shape[1]
print(f"[공선성] {d_var} → {d_filt} 비트 ({int(drop_mask.sum())}개 제거, |r|>0.9 중복)")
print(f"=> 최종: ECFP raw {d_raw} → filtered {d_filt} "
      f"({100 * (1 - d_filt / d_raw):.1f}% 축소)")


In [ ]:
# 필터링 단계별 차원 축소 막대
stages = ["raw\n(2048)", "저분산\n필터 후", "공선성\n필터 후"]
dvals = [d_raw, d_var, d_filt]
fig, ax = plt.subplots(figsize=(6.8, 4.6))
bars = ax.bar(stages, dvals, color=["#E69F00", "#56B4E9", "#009E73"],
              edgecolor="black", linewidth=0.4)
ax.set_ylabel("특징(비트) 수"); ax.set_title("ECFP 특징 필터링: 단계별 차원 축소")
for b, v in zip(bars, dvals):
    ax.text(b.get_x() + b.get_width() / 2, v + 20, str(v), ha="center", fontsize=11)
plt.tight_layout(); plt.show()

summary = pd.DataFrame({
    "단계": ["ECFP raw", "저분산 필터 후", "공선성 필터 후"],
    "차원": [d_raw, d_var, d_filt],
    "누적 제거": [0, d_raw - d_var, d_raw - d_filt],
    "raw 대비 축소율(%)": [0.0, round(100*(1-d_var/d_raw), 1), round(100*(1-d_filt/d_raw), 1)],
})
print("=== 특징 필터링 요약(실측) ===")
summary


## 8. 정리 — 좋은 특징이 모델링의 출발점

- **특징 생성**: 같은 분자라도 표현이 다릅니다. **기술자**(해석 가능·저차원)와 **ECFP 지문**(고차원·희소)을 모두 만들어 두면 이후 모델링 선택지가 넓어집니다.
- **특징 분석(EDA)**: 분포·타깃 상관·공선성 히트맵·화학공간 투영·ECFP 비트 통계를 통해 특징이 **무엇을 담고 있고, 어디가 중복/무의미한지**를 눈으로 확인했습니다. 이 이해가 없으면 필터링과 모델 선택이 근거 없는 추측이 됩니다.
- **특징 필터링**: 저분산+공선성 필터로 ECFP 차원을 크게 줄였습니다(위 표 실측). 필터는 **train에서만 학습**해 누수를 막았습니다 — 이 습관이 모델링 단계에서 정직한 성능 평가로 이어집니다.

> ⚠️ 교육용 데모입니다. 모든 통계·그림은 이 노트북이 ESOL 실데이터로 실제 계산한 값이며, `logS`는 특징을 이해하기 위한 분석 대상으로만 사용했습니다(모델 학습 없음).

**좋은 특징을 만들고 이해하는 것이 모델링의 출발점입니다 — 다음 [02 노트북]에서는 여기서 만든 특징(기술자·ECFP filtered·ChemBERTa 임베딩)으로 실제 QSAR 회귀 모델을 학습하고 성능을 비교합니다.**

**참고문헌 (실재)**
- Delaney. *ESOL: Estimating Aqueous Solubility Directly from Molecular Structure.* J Chem Inf Comput Sci 44:1000–1005 (2004).
- Rogers & Hahn. *Extended-Connectivity Fingerprints.* J Chem Inf Model 50:742–754 (2010).
- Todeschini & Consonni. *Handbook of Molecular Descriptors.* Wiley-VCH (2000).
- RDKit: Open-source cheminformatics (https://www.rdkit.org)